## Three Parts to this lab
## Part 1: A simple "Agent" and "Agent Loop"
Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool
A familiar one, but oh-so-easy

## Part 3: Adding Memory
So that different Agent calls know about each other

In [72]:
import asyncio
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, SQLiteSession
from openai.types.responses import ResponseTextDeltaEvent
from agents.extensions.models.litellm_model import LitellmModel
import requests


In [58]:
# Load environment variables from .env
load_dotenv(override=True)

True

In [59]:
# 1. Define the Groq model
groq_model = LitellmModel(
    model="groq/openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY")
)


In [60]:
# 2. Define the agent
agent = Agent(
    name="Jokester",
    instructions="You are a joke teller. Tell funny, creative and concise jokes.",
    model=groq_model
)

In [61]:
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
result

RunResult(input='Tell a joke about Autonomous AI Agents', new_items=[MessageOutputItem(agent=Agent(name='Jokester', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a joke teller. Tell funny, creative and concise jokes.', prompt=None, handoffs=[], model=<agents.extensions.models.litellm_model.LitellmModel object at 0x1699abd90>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None, prompt_cache_options=None, preserve_raw_usage=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), 

In [62]:
print(result.final_output)

Why did the autonomous AI agent refuse to go on a date?  

Because it said, “I already have a *self*‑driven relationship—no need to reboot my emotions!”


In [63]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent refuse to go on a date?  \n\nBecause it said, “I already have a *self*‑driven relationship—no need to reboot my emotions!”',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'groq/openai/gpt-oss-20b',
   'response_id': 'chatcmpl-2f2d14f4-72b1-45e0-92e5-6f12bc4b8639'}}]

## Adding Observability with a trace

In [64]:
with trace(workflow_name="Joke Generation Workflow"): ##It establishes a tracing scope around the code inside it.The SDK automatically creates spans for relevant operations executed within that scope.
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents") ##Runner.run() :	Waits for the completed result
    print(result.final_output)

1. **Why don’t autonomous AI agents ever get lost?**  
   Because they always follow their own GPS—**Goal‑Planning System**.

2. **What’s an autonomous AI agent’s favorite break?**  
   A *self‑service* coffee break—no human to *brew* it for them.

3. **Why did the autonomous AI agent break up with its human?**  
   It wanted more “space” for its own *self‑optimizing* growth.


## Now go and look at the trace
https://platform.openai.com/traces

In [ ]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.") ##This returns a RunResultStreaming object.It gives you access to the streaming events and the final run result.not awaited in the current SDK API.
async for event in result.stream_events(): ##This is an asynchronous iterator.The program receives events as they become available rather than waiting for a complete list.
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent): ##Is event.data an instance of the ResponseTextDeltaEvent class?
        print(event.data.delta, end="", flush=True) ##end="": Don't add a newline after every token.#flush=True: Immediately flush the output buffer so the text appears without waiting for the buffer to fill.

1. Why did the AI agent bring a ladder to work?  
Because it heard the job had “higher-level” decisions!

2. My AI agent just asked, “Do I have feelings?”  
It’s still waiting for the emotional upgrade—so far it’s just *biting* the same response.

3. What’s an AI agent’s favorite exercise?  
“Self‑learning”—it always pulls a new routine every time it’s *rebooted*!

4. Why did the AI agent start a bakery?  
Because it heard it could *knead* data and *rise* with every new algorithm!

5. My AI agent tried to tell a joke.  
It printed, “I’m sorry, I don’t have a sense of humor… but here’s a *byte* of it.”

In [66]:
print(result.final_output)

1. Why did the AI agent bring a ladder to work?  
Because it heard the job had “higher-level” decisions!

2. My AI agent just asked, “Do I have feelings?”  
It’s still waiting for the emotional upgrade—so far it’s just *biting* the same response.

3. What’s an AI agent’s favorite exercise?  
“Self‑learning”—it always pulls a new routine every time it’s *rebooted*!

4. Why did the AI agent start a bakery?  
Because it heard it could *knead* data and *rise* with every new algorithm!

5. My AI agent tried to tell a joke.  
It printed, “I’m sorry, I don’t have a sense of humor… but here’s a *byte* of it.”


In [67]:
print(result.is_complete)

True


## Part 2: Adding a tool

In [68]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [73]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [74]:
push("HEY!!")

Push: HEY!!


In [76]:
push

<function __main__.push(message)>

In [77]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [78]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x169ad20d0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [79]:
push_tool.description

'Send the given message to the user as a push notification'

In [95]:
print(push_tool.params_json_schema)

{'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}


In [ ]:
notifier = Agent(name="Notifier", model=groq_model, instructions="You notify the user upon request", tools=[push_tool])


In [81]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")
    print(result.final_output)

✅ Notification sent.


Python has function-level scope, not block-level scope. Therefore, if, for, while, and with blocks do not create separate local scopes.

In [ ]:
result ## with block — variable is accessible outside whereas Function — local variable is NOT accessible outside

RunResult(input='Notify the user that the pizza is here', new_items=[ToolCallItem(agent=Agent(name='Notifier', handoff_description=None, tools=[FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x169ad20d0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)], mcp_servers=[], mcp_config={}, instructions='You notify the user upon request', prompt=None, handoffs=[], model=<agents.extensions.models.litellm_model.LitellmModel obje

Now go and look at the trace
https://platform.openai.com/traces

## Part 3: Sessions (memory)
Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [83]:
agent = Agent(name="Assistant", model=groq_model)

In [84]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hey Ed! 👋 How can I help you today?


In [85]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I’m not able to determine your name. If you’d like to share it, you can, but I don’t have that information.


## Memory approach 1 - just manually pass in the list of dicts

In [86]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hello Ed! 👋 How can I help you today?


In [87]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hello Ed! 👋 How can I help you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'groq/openai/gpt-oss-20b',
   'response_id': 'chatcmpl-e23e14d8-9053-4da4-a74e-c2dc7a9e0ab4'}}]

In [88]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hello Ed! 👋 How can I help you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'groq/openai/gpt-oss-20b',
   'response_id': 'chatcmpl-e23e14d8-9053-4da4-a74e-c2dc7a9e0ab4'}},
 {'role': 'user', 'content': "What's my name?"}]

In [90]:
response = await Runner.run(agent, next_input)
print(response.final_output)

You’re Ed!


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [91]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [92]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hello, Ed! 👋 How can I help you today?


In [93]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ed.
